[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_DMTest/VaR_CEE_DMTest.ipynb)

# VaR_CEE_DMTest

Performs pairwise Diebold-Mariano tests using asymmetric tick (quantile) loss at the 1% VaR level
across all 10 models and 10 CEE series. Pools tick losses across series, computes DM statistics
with Harvey et al. small-sample correction, and produces a p-value heatmap with significance annotations.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from scipy import stats

%matplotlib inline

# ── Configuration ──────────────────────────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"},
}

MODELS = [
    'HS', 'GJR-GARCH', 'ARIMA-Conformal', 'LSTM-Conformal',
    'Chronos-2', 'TimesFM-2.5', 'Moirai-2.0',
    'Chronos-2-Conf', 'TimesFM-2.5-Conf', 'Moirai-2.0-Conf'
]

SHORT_NAMES = [
    'HS', 'GJR-GARCH', 'ARIMA-Conf', 'LSTM-Conf',
    'Chronos-2', 'TimesFM-2.5', 'Moirai-2.0',
    'Chronos-2-C', 'TimesFM-2.5-C', 'Moirai-2.0-C'
]

SERIES = []
for info in MARKETS.values():
    SERIES.append(f"{info['index']}_ret")
    SERIES.append(f"{info['fx']}_ret")

ALPHA = 0.01

VAR_DIR = Path("var_forecasts")
BT_DIR = Path("backtesting")
VAR_DIR.mkdir(exist_ok=True)
BT_DIR.mkdir(exist_ok=True)

## Upload VaR Forecast CSVs

Upload ALL model forecast CSV files (up to 100 files). Each file is named `{Model}_{Series}.csv`
and must contain columns: `date`, `y_true`, `var_01`.

In [ ]:
from google.colab import files
print("Upload all VaR forecast CSV files:")
uploaded = files.upload()
for fname, content in uploaded.items():
    dest = VAR_DIR / fname
    with open(dest, "wb") as f:
        f.write(content)
print(f"\nUploaded {len(uploaded)} files to {VAR_DIR}")

## Tick Loss and Diebold-Mariano Test Functions

The **tick (quantile) loss** is the scoring rule for VaR forecasts:

$$L_\alpha(r_t, \hat{q}_t) = (\alpha - \mathbf{1}_{r_t < \hat{q}_t}) \cdot (r_t - \hat{q}_t)$$

The **Diebold-Mariano test** (Diebold & Mariano, 1995) compares two forecast methods via the loss differential $d_t = L_1(t) - L_2(t)$. Under $H_0$: equal predictive accuracy, the test statistic

$$DM = \frac{\bar{d}}{\sqrt{\hat{V}(\bar{d})}}$$

is asymptotically standard normal. We apply the **Harvey et al. (1997)** finite-sample correction and use the $t$-distribution for inference.

In [ ]:
def tick_loss(returns, var_forecasts, alpha):
    """Asymmetric tick / quantile loss."""
    indicator = (returns < var_forecasts).astype(float)
    return (alpha - indicator) * (returns - var_forecasts)


def diebold_mariano(loss1, loss2, h=1):
    """Two-sided DM test with Harvey et al. (1997) small-sample correction."""
    d = loss1 - loss2
    T = len(d)
    d_bar = np.mean(d)
    gamma_0 = np.var(d, ddof=1)
    V_d = max(gamma_0 / T, 1e-20)
    DM = d_bar / np.sqrt(V_d)
    correction = np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
    DM_adj = DM * correction
    p_value = 2 * stats.t.sf(np.abs(DM_adj), df=T-1)
    return float(DM_adj), float(p_value)


def load_var(model, series):
    """Load a single model/series VaR forecast CSV."""
    fp = VAR_DIR / f"{model}_{series}.csv"
    return pd.read_csv(fp, parse_dates=['date'])

## Compute Pooled Tick Losses

For each model, tick losses are computed per series and then **pooled (concatenated)** across all 10 CEE series before running the DM test. This gives a single aggregate comparison per model pair.

In [ ]:
n_models = len(MODELS)
pooled_losses = {m: [] for m in MODELS}

for series in SERIES:
    dfs = {}
    for m in MODELS:
        dfs[m] = load_var(m, series)

    # Align on common dates
    common_dates = dfs[MODELS[0]]['date']
    for m in MODELS[1:]:
        common_dates = pd.merge(common_dates, dfs[m][['date']], on='date')['date']

    returns = None
    var_dict = {}
    for m in MODELS:
        df_a = dfs[m][dfs[m]['date'].isin(common_dates)].sort_values('date').reset_index(drop=True)
        if returns is None:
            returns = df_a['y_true'].values
        var_dict[m] = df_a['var_01'].values

    for m in MODELS:
        tl = tick_loss(returns, var_dict[m], ALPHA)
        pooled_losses[m].append(tl)

for m in MODELS:
    pooled_losses[m] = np.concatenate(pooled_losses[m])

print(f"Pooled tick-loss vector length per model: {len(pooled_losses[MODELS[0]])}")
print(f"Number of series pooled: {len(SERIES)}")

## Pairwise Diebold-Mariano Tests

Compute the DM statistic and p-value for all $\binom{10}{2} = 45$ model pairs and store them in a matrix.

In [ ]:
dm_stat_matrix = np.zeros((n_models, n_models))
dm_pval_matrix = np.ones((n_models, n_models))

for i in range(n_models):
    for j in range(i+1, n_models):
        stat, pval = diebold_mariano(pooled_losses[MODELS[i]], pooled_losses[MODELS[j]])
        dm_stat_matrix[i, j] = stat
        dm_stat_matrix[j, i] = -stat
        dm_pval_matrix[i, j] = pval
        dm_pval_matrix[j, i] = pval

# Save results to CSV
rows = []
for i in range(n_models):
    for j in range(i+1, n_models):
        rows.append({
            'model_i': MODELS[i],
            'model_j': MODELS[j],
            'DM_stat': round(dm_stat_matrix[i, j], 4),
            'p_value': round(dm_pval_matrix[i, j], 6),
            'sig_10': dm_pval_matrix[i, j] < 0.10,
            'sig_05': dm_pval_matrix[i, j] < 0.05,
            'sig_01': dm_pval_matrix[i, j] < 0.01,
            'mean_loss_i': round(np.mean(pooled_losses[MODELS[i]]), 8),
            'mean_loss_j': round(np.mean(pooled_losses[MODELS[j]]), 8),
        })

dm_df = pd.DataFrame(rows)
dm_path = BT_DIR / "dm_tick_loss_1pct.csv"
dm_df.to_csv(dm_path, index=False)
print(f"Saved {len(dm_df)} pairwise comparisons to {dm_path}")
dm_df.head(10)

## Model Ranking by Mean Tick Loss

In [ ]:
print("Model ranking by mean tick loss (lower = better):\n")
model_losses = [(m, np.mean(pooled_losses[m])) for m in MODELS]
model_losses.sort(key=lambda x: x[1])
for rank, (m, loss) in enumerate(model_losses, 1):
    print(f"  {rank:2d}. {m:20s}  {loss:.8f}")

## Key Result: TimesFM-2.5 vs GJR-GARCH

In [ ]:
idx_tfm = MODELS.index('TimesFM-2.5')
idx_garch = MODELS.index('GJR-GARCH')
stat_tg = dm_stat_matrix[idx_tfm, idx_garch]
pval_tg = dm_pval_matrix[idx_tfm, idx_garch]
mean_tfm = np.mean(pooled_losses['TimesFM-2.5'])
mean_garch = np.mean(pooled_losses['GJR-GARCH'])

print("KEY RESULT: TimesFM-2.5 vs GJR-GARCH")
print(f"  Mean tick loss TimesFM-2.5: {mean_tfm:.8f}")
print(f"  Mean tick loss GJR-GARCH:   {mean_garch:.8f}")
print(f"  DM statistic:              {stat_tg:.4f}")
print(f"  p-value:                    {pval_tg:.6f}")
if pval_tg < 0.01:
    print(f"  => SIGNIFICANT at 1% level (***)")
elif pval_tg < 0.05:
    print(f"  => SIGNIFICANT at 5% level (**)")
elif pval_tg < 0.10:
    print(f"  => SIGNIFICANT at 10% level (*)")
else:
    print(f"  => NOT statistically significant")

if stat_tg < 0:
    print(f"  => TimesFM-2.5 has LOWER average tick loss (better)")
else:
    print(f"  => GJR-GARCH has LOWER average tick loss (better)")

## Figure 10: DM Test Heatmap

Visualizes all pairwise p-values as a heatmap. Cells are colored red (significant) to green (not significant). DM statistics are annotated with significance stars: \* p<0.10, \*\* p<0.05, \*\*\* p<0.01.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

pval_display = dm_pval_matrix.copy()
np.fill_diagonal(pval_display, np.nan)

cmap = plt.cm.RdYlGn_r
masked = np.ma.masked_where(np.isnan(pval_display), pval_display)

im = ax.imshow(masked, cmap=cmap, vmin=0, vmax=0.20, aspect='equal')

for i in range(n_models):
    for j in range(n_models):
        if i == j:
            ax.text(j, i, '--', ha='center', va='center', fontsize=7, color='gray')
            continue
        pval = dm_pval_matrix[i, j]
        if pval < 0.01:
            stars = '***'
        elif pval < 0.05:
            stars = '**'
        elif pval < 0.10:
            stars = '*'
        else:
            stars = ''

        stat_val = dm_stat_matrix[i, j]
        text_color = 'white' if pval < 0.05 else 'black'
        ax.text(j, i - 0.15, f'{stat_val:.2f}', ha='center', va='center',
                fontsize=6.5, color=text_color)
        if stars:
            ax.text(j, i + 0.2, stars, ha='center', va='center',
                    fontsize=7, color=text_color, fontweight='bold')

ax.set_xticks(range(n_models))
ax.set_xticklabels(SHORT_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(n_models))
ax.set_yticklabels(SHORT_NAMES, fontsize=8)

fig.colorbar(im, ax=ax, shrink=0.8, label='p-value')

ax.set_title('Figure 10. Diebold-Mariano Test Heatmap (Tick Loss, 1% VaR)\n'
             'DM statistic shown; * p<0.10, ** p<0.05, *** p<0.01')

plt.tight_layout()
fig.savefig('fig10_dm_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved fig10_dm_heatmap.png")

## Download Results

In [ ]:
import zipfile, io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(dm_path, "dm_tick_loss_1pct.csv")
    zf.write("fig10_dm_heatmap.png", "fig10_dm_heatmap.png")
zip_buffer.seek(0)
with open("dm_test_results.zip", "wb") as fout:
    fout.write(zip_buffer.read())
files.download("dm_test_results.zip")